* [1. Import Data](#import-data)
* [2. Data Preprocessing](#data-preprocessing)
    * [2.1. Train-Test split](#train-test-split)
    * [2.2. Preprocessing Pipeline: One-hot Encoding and Standardization](#preprocessing-pipeline%3A-one-hot-encoding-and-standardization)
* [3. XGBoost + OneVsRestClassifier](#xgboost-%2B-onevsrestclassifier)
* [4. XGBoost + MultiOutputClassifier](#xgboost-%2B-multioutputclassifier)
* [5. XGBoost + ClassifierChain](#xgboost-%2B-classifierchain)
* [6. Retrain on full dataset and Submit Predictions](#retrain-on-full-dataset-and-submit-predictions)

In [1]:
import os, sys

import numpy as np
from scipy.stats import chi2_contingency
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

<a id="import-data"></a>
# 1. Import Data

In [2]:
df = pd.read_csv('/kaggle/input/lish-moa/train_features.csv')
df.head()

,sig_id,cp_type,cp_time,cp_dose,g-0,g-1,g-2,g-3,g-4,g-5,...,c-90,c-91,c-92,c-93,c-94,c-95,c-96,c-97,c-98,c-99
0,id_000644bb2,trt_cp,24,D1,1.0620,0.5577,-0.2479,-0.6208,-0.1944,-1.0120,...,0.2862,0.2584,0.8076,0.5523,-0.1912,0.6584,-0.3981,0.2139,0.3801,0.4176
1,id_000779bfc,trt_cp,72,D1,0.0743,0.4087,0.2991,0.0604,1.0190,0.5207,...,-0.4265,0.7543,0.4708,0.0230,0.2957,0.4899,0.1522,0.1241,0.6077,0.7371
2,id_000a6266a,trt_cp,48,D1,0.6280,0.5817,1.5540,-0.0764,-0.0323,1.2390,...,-0.7250,-0.6297,0.6103,0.0223,-1.3240,-0.3174,-0.6417,-0.2187,-1.4080,0.6931
3,id_0015fd391,trt_cp,48,D1,-0.5138,-0.2491,-0.2656,0.5288,4.0620,-0.8095,...,-2.0990,-0.6441,-5.6300,-1.3780,-0.8632,-1.2880,-1.6210,-0.8784,-0.3876,-0.8154
4,id_001626bd3,trt_cp,72,D2,-0.3254,-0.4009,0.9700,0.6919,1.4180,-0.8244,...,0.0042,0.0048,0.6670,1.0690,0.5523,-0.3031,0.1094,0.2885,-0.3786,0.7125


In [3]:
df.shape

(23814, 876)

In [4]:
df_targets = pd.read_csv('../input/lish-moa/train_targets_scored.csv')
df_targets.head()

,sig_id,5-alpha_reductase_inhibitor,11-beta-hsd1_inhibitor,acat_inhibitor,acetylcholine_receptor_agonist,acetylcholine_receptor_antagonist,acetylcholinesterase_inhibitor,adenosine_receptor_agonist,adenosine_receptor_antagonist,adenylyl_cyclase_activator,...,tropomyosin_receptor_kinase_inhibitor,trpv_agonist,trpv_antagonist,tubulin_inhibitor,tyrosine_kinase_inhibitor,ubiquitin_specific_protease_inhibitor,vegfr_inhibitor,vitamin_b,vitamin_d_receptor_agonist,wnt_inhibitor
0,id_000644bb2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,id_000779bfc,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,id_000a6266a,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,id_0015fd391,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,id_001626bd3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
df_targets.shape

(23814, 207)

In [6]:
df_test_sub = pd.read_csv('/kaggle/input/lish-moa/test_features.csv')

## 1.1. Find categorical columns and change their *Dtype* from `object` to `Categorical`

In [7]:
def summarize_categoricals(df, show_levels=False, threshold=5):
    """
        Display uniqueness in each column
    """
    data = [[df[c].unique(), len(df[c].unique()), df[c].isnull().sum()] for c in df.columns]
    df_temp = pd.DataFrame(data, index=df.columns,
                           columns=['Levels', 'No. of Levels', 'No. of Missing Values'])
    return df_temp[df_temp['No. of Levels'] <= threshold].iloc[:, 0 if show_levels else 1:]


def return_categoricals(df, threshold=5):
    """
        Returns a list of columns that have less than or equal to
        `threshold` number of unique categorical levels
    """
    return list(filter(lambda c: c if len(df[c].unique()) <= threshold else None,
                       df.columns))


def to_categorical(columns, df):
    """
        Converts the columns passed in `columns` to categorical datatype
    """
    for col in columns:
        df[col] = df[col].astype('category')
    return df

In [8]:
summarize_categoricals(df, show_levels=True, threshold=500)

,Levels,No. of Levels,No. of Missing Values
cp_type,"[trt_cp, ctl_vehicle]",2,0
cp_time,"[24, 72, 48]",3,0
cp_dose,"[D1, D2]",2,0


In [9]:
categorical_columns = return_categoricals(df)

In [10]:
df = to_categorical(categorical_columns, df)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23814 entries, 0 to 23813
Columns: 876 entries, sig_id to c-99
dtypes: category(3), float64(872), object(1)
memory usage: 158.7+ MB


In [12]:
df_test_sub = to_categorical(categorical_columns, df_test_sub)

<a id="data-preprocessing"></a>
# 2. Data Preprocessing

In [13]:
x = df.iloc[:, 1:]
y = df_targets.iloc[:, 1:]

categorical_columns = list(x.select_dtypes(include='category').columns)
numeric_columns = list(x.select_dtypes(exclude='category').columns)

<a id="train-test-split"></a>
## 2.1. Train-Test split

In [14]:
from sklearn.model_selection import train_test_split

data_splits = train_test_split(x, y, test_size=0.15, random_state=0,
                               shuffle=True)
x_train, x_test, y_train, y_test = data_splits

list(map(lambda x: x.shape, [x, y, x_train, x_test,
                             y_train, y_test]))

[(23814, 875),
 (23814, 206),
 (20241, 875),
 (3573, 875),
 (20241, 206),
 (3573, 206)]

<a id="preprocessing-pipeline%3A-one-hot-encoding-and-standardization"></a>
## 2.2. Preprocessing Pipeline: One-hot Encoding and Standardization

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline 


numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', dtype=np.int))])

## Column Transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_columns),
        ('cat', categorical_transformer, categorical_columns)],
    remainder='passthrough')


## Applying Column Transformer
x_train = preprocessor.fit_transform(x_train)
x_test = preprocessor.transform(x_test)

x_test_sub = preprocessor.transform(df_test_sub.iloc[:, 1:])


## Label encoding
y_train = y_train.to_numpy(dtype=np.int64)
y_test = y_test.to_numpy(dtype=np.int64)


## Save feature names after one-hot encoding for feature importances plots
feature_names = list(preprocessor.named_transformers_['cat'].named_steps['onehot'] \
                     .get_feature_names(input_features=categorical_columns))
feature_names = feature_names + numeric_columns

<a id="xgboost-%2B-onevsrestclassifier"></a>
# 3. XGBoost + OneVsRestClassifier

In [16]:
from sklearn.metrics import log_loss
from xgboost import XGBClassifier
from sklearn.multiclass import OneVsRestClassifier

xgb = XGBClassifier(tree_method='gpu_hist',
                    predictor='gpu_predictor',
                    random_state=0, n_jobs=-1)

ovr_clf = OneVsRestClassifier(estimator=xgb, n_jobs=-1)

ovr_clf.fit(x_train, y_train)

ovr_probs = ovr_clf.predict_proba(x_test)

log_loss(y_test, ovr_probs)

2.7283924654707747

<a id="xgboost-%2B-multioutputclassifier"></a>
# 4. XGBoost + MultiOutputClassifier

In [17]:
from sklearn.multioutput import MultiOutputClassifier

mo_clf = MultiOutputClassifier(estimator=xgb, n_jobs=-1)

mo_clf.fit(x_train, y_train)

mo_probs = mo_clf.predict_proba(x_test)

n_classes = y_test.shape[1]
n_test_samples = x_test.shape[0]
mo_probs_pos = np.zeros((n_test_samples, n_classes))

for c in range(n_classes):
    c_probs = mo_probs[c]
    mo_probs_pos[:, c] = c_probs[:, 1]

log_loss(y_test, mo_probs_pos)

2.7225397551458514

<a id="xgboost-%2B-classifierchain"></a>
# 5. XGBoost + ClassifierChain

In [18]:
from sklearn.multioutput import ClassifierChain
from joblib import Parallel, delayed
import timeit

chains = [ClassifierChain(base_estimator=xgb, order='random')
          for i in range(5)]

chains = Parallel(n_jobs=-1)(delayed(chain.fit)(x_train, y_train) for chain in chains)

chains_ensemble_proba = Parallel(n_jobs=-1)(delayed(chain.predict_proba)(x_test) for chain in chains)

log_loss(y_test, np.array(chains_ensemble_proba).mean(axis=0))

/opt/conda/lib/python3.7/site-packages/joblib/externals/loky/process_executor.py:706: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  "timeout or by a memory leak.", UserWarning


2.6462115617484008

<a id="retrain-on-full-dataset-and-submit-predictions"></a>
# 6. Retrain on full dataset and Submit Predictions

In [19]:
## Final Model
x_train_full = preprocessor.fit_transform(x)
y_train_full = y.to_numpy(dtype=np.int64)

x_test_final = preprocessor.transform(df_test_sub.iloc[:, 1:])

chains = Parallel(n_jobs=-1)(delayed(chain.fit)(x_train_full, y_train_full) for chain in chains)

final_proba = Parallel(n_jobs=-1)(delayed(chain.predict_proba)(x_test_final) for chain in chains)

In [20]:
pd.DataFrame(np.array(final_proba).mean(axis=0),
             index=df_test_sub['sig_id'],
             columns=df_targets.columns[1:]).to_csv('submission.csv')

Thank You!!